In [1]:
import requests
import json
import os
import numpy as np

In [2]:
token = json.load(open(os.path.join(os.path.expanduser("~"), "ibm_api.json"), "r"))["apikey"]
# token = open(os.path.join(os.path.expanduser("~"), "token.txt"), "r")

In [3]:
iam_response = requests.post(
    "https://iam.cloud.ibm.com/identity/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data=f"grant_type=urn:ibm:params:oauth:grant-type:apikey&apikey={token}",
)
access_token = iam_response.json()["access_token"]

In [4]:
reqUrl = "https://quantum.cloud.ibm.com/api/v1/backends"
 
headersList = {
  "Accept": "application/json",
  "Authorization": "Bearer {}".format(access_token),
  "Service-CRN": "crn:v1:bluemix:public:quantum-computing:us-east:a/ca6dbe7ad4314f858ee5e43105e2ab0e:70045ff3-46fb-4495-9692-b7d2553f43ed::",
  "IBM-API-Version": "2026-02-15"
}
 
payload = ""
 
response = requests.request("GET", reqUrl, data=payload,  headers=headersList)
 
print(response.json())

{'devices': [{'name': 'ibm_fez', 'status': {'name': 'online', 'reason': 'available'}, 'qubits': 156, 'clops': {'type': 'hardware', 'value': 320000}, 'processor_type': {'family': 'Heron', 'revision': '2'}, 'queue_length': 2, 'performance_metrics': {'two_q_error_best': {'gate': 'cz', 'qubits': [91, 98], 'unit': '', 'value': 0.0014688022}, 'two_q_error_layered': {'unit': '', 'value': 0.0049178777}, 'two_q_error_median': {'unit': '', 'value': 0.0026832712}, 'readout_error_median': {'unit': '', 'value': 0.014282227}}}, {'name': 'ibm_torino', 'status': {'name': 'online', 'reason': 'available'}, 'qubits': 133, 'clops': {'type': 'hardware', 'value': 290000}, 'processor_type': {'family': 'Heron', 'revision': '1'}, 'queue_length': 0, 'performance_metrics': {'two_q_error_best': {'gate': 'cz', 'qubits': [25, 26], 'unit': '', 'value': 0.0011137852}, 'two_q_error_layered': {'unit': '', 'value': 0.0066505484}, 'two_q_error_median': {'unit': '', 'value': 0.002458021}, 'readout_error_median': {'unit': 

In [5]:
response = requests.request(
    "GET",
    "https://quantum.cloud.ibm.com/api/v1/backends/ibm_fez/properties",
    headers={
        "Accept": "application/json",
        "IBM-API-Version": "2026-02-15",
        "Authorization": "Bearer {token}".format(token=access_token),
        "Service-CRN": "crn:v1:bluemix:public:quantum-computing:us-east:a/ca6dbe7ad4314f858ee5e43105e2ab0e:70045ff3-46fb-4495-9692-b7d2553f43ed::"
    },
)

In [6]:
json.dump(response.json(), open("backend_data.json", "w"), indent=4)

In [7]:
data = response.json()

In [8]:
data.keys()

dict_keys(['backend_name', 'backend_version', 'gates', 'general', 'general_qlists', 'last_update_date', 'qubits'])

In [9]:
import pandas as pd

In [13]:
basis_gates = [["x", "sx", "rx", "rz"], ["cz", "rzz"]]

In [63]:
column_names = ["Qubits", "T1 (us)", "T2 (us)", "Prob meas 0 prep 1", "Prob meas 1 prep 0", "Single Qubit Gate Length (ns)"]
for gate in basis_gates[0]:
    column_names.append(f"{gate} Error")
column_names.append("Gate Length (ns)")
for gate in basis_gates[1]:
    column_names.append(f"{gate} Gate Length (ns)")
    column_names.append(f"{gate} Error")

In [64]:
df = pd.DataFrame(columns=column_names)

In [65]:
qubits = []
t1 = []
t2 = []
prob01 = []
prob10 = []
for q, items in enumerate(data["qubits"]):
    qubits.append(q)
    for entry in items:
        if entry["name"] == "T1":
            t1.append(entry["value"])
        elif entry["name"] == "T2":
            t2.append(entry["value"])
        elif entry["name"] == "prob_meas0_prep1":
            prob01.append(entry["value"])
        elif entry["name"] == "prob_meas1_prep0":
            prob10.append(entry["value"])
        else:
            continue
df["Qubits"] = qubits
df["T1 (us)"] = t1
df["T2 (us)"] = t2
df["Prob meas 0 prep 1"] = prob01
df["Prob meas 1 prep 0"] = prob10

In [57]:
data["gates"][156*5]

{'gate': 'cz',
 'name': 'cz72_73',
 'parameters': [{'date': '2026-02-27T15:04:55.213622Z',
   'name': 'gate_error',
   'unit': '',
   'value': 1},
  {'date': '2026-02-27T15:04:55.213622Z',
   'name': 'gate_length',
   'unit': 'ns',
   'value': 68}],
 'qubits': [72, 73]}

In [66]:
for two_q in basis_gates[1]:
    df.loc[:, "{} Error".format(two_q)] = ""
    df.loc[:, "{} Gate Length (ns)".format(two_q)] = ""
for item in data["gates"]:
    if item["gate"] in basis_gates[0]:
        col_name = f"{item['gate']} Error"
        error_rate = item["parameters"][0]["value"]
        gate_length = item["parameters"][1]["value"]
        qubit_index = item["qubits"][0]
        df.loc[qubit_index, col_name] = error_rate
        df.loc[qubit_index, "Single Qubit Gate Length (ns)"] = gate_length
    elif item["gate"] in basis_gates[1]:
        col_name = f"{item['gate']} Error"
        control_qubit = item["qubits"][0]
        target_qubit = item["qubits"][1]
        error_rate = item["parameters"][0]["value"] if item["parameters"][0]["name"] == "gate_error" else np.nan
        gate_length = item["parameters"][1]["value"] if item["parameters"][1]["name"] == "gate_length" else np.nan
        gate_length_string = f"{target_qubit}:{gate_length};"
        error_rate_string = f"{target_qubit}:{error_rate};"
        df.loc[control_qubit, col_name] += error_rate_string
        df.loc[control_qubit, f"{item['gate']} Gate Length (ns)"] += gate_length_string
    else:
        continue

In [67]:
df.loc[:, "Gate Length (ns)"] = ""
for gate in basis_gates[1]:
    for qubit in range(len(df)):
        gate_length_value = df["{} Gate Length (ns)".format(gate)][qubit]
        error_rate_value = df["{} Error".format(gate)][qubit]
        if gate_length_value == "":
            pass
        else:
            gate_length_value = gate_length_value[:-1]
        if error_rate_value == "":
            pass
        else:
            error_rate_value = error_rate_value[:-1]
        df.loc[qubit, "Gate Length (ns)"] = gate_length_value
        df.loc[qubit, "{} Error".format(gate)] = error_rate_value

In [69]:
drop_cols = []
for gate in basis_gates[1]:
    drop_cols.append("{} Gate Length (ns)".format(gate))
df.drop(columns=drop_cols, inplace=True)

In [70]:
df.head(5)

,Qubits,T1 (us),T2 (us),Prob meas 0 prep 1,Prob meas 1 prep 0,Single Qubit Gate Length (ns),x Error,sx Error,rx Error,rz Error,Gate Length (ns),cz Error,rzz Error
0,0,49.771128,19.150684,0.038086,0.001709,24,0.001536,0.001536,0.001536,0,1:68,1:0.009493429650216878,1:0.010292408774270095
1,1,176.456738,229.036299,0.029053,0.022217,24,0.000408,0.000408,0.000408,0,2:68;0:68,2:0.0024821634053313057;0:0.009493429650216878,2:0.0020323276724973915;0:0.010292408774270095
2,2,245.274361,175.727365,0.012207,0.008789,24,0.002911,0.002911,0.002911,0,1:68;3:68,1:0.0024821634053313057;3:0.002129426890814773,1:0.0020323276724973915;3:0.0019513819362031726
3,3,223.539450,197.808434,0.050293,0.059082,24,0.00018,0.00018,0.00018,0,16:68;2:68;4:68,16:0.0017028120555028503;2:0.00212942689081477...,16:0.0018286108815005198;2:0.00195138193620317...
4,4,220.345742,52.077188,0.014404,0.003662,24,0.000245,0.000245,0.000245,0,3:68;5:68,3:0.0019060087877015297;5:0.0028739708515657103,3:0.001886916006222622;5:0.0020825521809950953
